# HealthCost AI — Modeling

**Goal:** Train and compare candidate models inside a single sklearn Pipeline 
(preprocessing + model), select the final model based on test-set metrics, and 
prepare it for saving as the deployable artifact.

**Input:** data/processed/X_train.csv, X_test.csv, y_train.csv, y_test.csv (from notebook 02)
**Candidates:** Linear Regression (baseline) → Random Forest Regressor → Gradient Boosting (if justified)
**Metrics:** MAE, RMSE, R² — evaluated on log-scale AND on original charges scale (after expm1)

## Load Data & Rebuild Preprocessor

In [1]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(1069, 6) (268, 6) (1069,) (268,)


## Baseline: Linear Regression

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

lr_pipeline.fit(X_train, y_train)
y_pred_log = lr_pipeline.predict(X_test)

# metrics on log scale
mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log = r2_score(y_test, y_pred_log)

# metrics on original charges scale
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred_log)
mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)
rmse_orig = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
r2_orig = r2_score(y_test_orig, y_pred_orig)

print(f"Log scale  -> MAE: {mae_log:.4f}, RMSE: {rmse_log:.4f}, R2: {r2_log:.4f}")
print(f"Orig scale -> MAE: {mae_orig:.2f}, RMSE: {rmse_orig:.2f}, R2: {r2_orig:.4f}")

Log scale  -> MAE: 0.2607, RMSE: 0.3978, R2: 0.8295
Orig scale -> MAE: 3755.92, RMSE: 7197.03, R2: 0.7181


## Random Forest Regressor

In [3]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
y_pred_log = rf_pipeline.predict(X_test)

mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log = r2_score(y_test, y_pred_log)

y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred_log)
mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)
rmse_orig = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
r2_orig = r2_score(y_test_orig, y_pred_orig)

print(f"Log scale  -> MAE: {mae_log:.4f}, RMSE: {rmse_log:.4f}, R2: {r2_log:.4f}")
print(f"Orig scale -> MAE: {mae_orig:.2f}, RMSE: {rmse_orig:.2f}, R2: {r2_orig:.4f}")

Log scale  -> MAE: 0.1911, RMSE: 0.3728, R2: 0.8502
Orig scale -> MAE: 2040.87, RMSE: 4362.51, R2: 0.8964


## Gradient Boosting Regressor

In [4]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor(n_estimators=200, random_state=42))
])

gb_pipeline.fit(X_train, y_train)
y_pred_log = gb_pipeline.predict(X_test)

mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log = r2_score(y_test, y_pred_log)

y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred_log)
mae_orig = mean_absolute_error(y_test_orig, y_pred_orig)
rmse_orig = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
r2_orig = r2_score(y_test_orig, y_pred_orig)

print(f"Log scale  -> MAE: {mae_log:.4f}, RMSE: {rmse_log:.4f}, R2: {r2_log:.4f}")
print(f"Orig scale -> MAE: {mae_orig:.2f}, RMSE: {rmse_orig:.2f}, R2: {r2_orig:.4f}")


Log scale  -> MAE: 0.1939, RMSE: 0.3398, R2: 0.8756
Orig scale -> MAE: 2314.62, RMSE: 4631.80, R2: 0.8832


## Model Comparison & Final Selection

| Model | MAE ($) | RMSE ($) | R² |
|---|---|---|---|
| Linear Regression | 3755.92 | 7197.03 | 0.7181 |
| Random Forest | **2040.87** | **4362.51** | **0.8964** |
| Gradient Boosting | 2314.62 | 4631.80 | 0.8832 |

**Decision: Random Forest Regressor** is selected as the final model.

Rationale:
- Best MAE/RMSE/R² on the original charges scale, which is the metric that matters for real-world 
  interpretation (dollar error), not just the log-transformed training scale.
- Gradient Boosting scored marginally better on log-scale RMSE/R² but worse on the original scale — 
  not a strong enough case to prefer it.
- Per the project's technology decisions, XGBoost is only justified if it clearly outperforms 
  RandomForest/GBM on this dataset. Since RF already outperforms GBM here, adding XGBoost would only 
  add complexity without a demonstrated benefit — skipped.

## Feature Importance (Sanity Check)
Should align with EDA findings: smoker and bmi expected to dominate.

In [5]:
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()
importances = rf_pipeline.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
importance_df = importance_df.sort_values('importance', ascending=False)
importance_df

,feature,importance
0,num__age,0.381089
5,cat__smoker_no,0.215904
6,cat__smoker_yes,0.210034
1,num__bmi,0.100098
2,num__children,0.049840
8,cat__region_northwest,0.008971
7,cat__region_northeast,0.008757
4,cat__sex_male,0.007323
3,cat__sex_female,0.006669
9,cat__region_southeast,0.006191


Note: `smoker` is one-hot encoded into two columns. Combined importance: 
`smoker_no + smoker_yes = 0.2159 + 0.2100 = 0.426` — the single most important feature overall, 
consistent with EDA (smoker was the dominant predictor). `age` (0.381) and `bmi` (0.100) follow. 
`sex` and `region` are negligible, matching earlier EDA findings.

## Save Final Model
Refit the pipeline on the full dataset (train+test) for the deployed model — 
test set was already used only for evaluation, not for the artifact we ship.
Saved together with a metadata file (version, date, metrics) per the project's 
model-versioning hygiene requirement.

In [6]:
import joblib
import json
from datetime import datetime

# refit on full data for deployment
X_full = pd.concat([X_train, X_test], ignore_index=True)
y_full = pd.concat([y_train, y_test], ignore_index=True)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])
final_pipeline.fit(X_full, y_full)

import os
os.makedirs('../models', exist_ok=True)
joblib.dump(final_pipeline, '../models/final_model.joblib')

metadata = {
    "model_type": "RandomForestRegressor",
    "version": "1.0.0",
    "trained_at": datetime.now().isoformat(),
    "target_transform": "log1p (inverse: expm1)",
    "features": ["age", "sex", "bmi", "children", "smoker", "region"],
    "test_set_metrics": {
        "mae_usd": 2040.87,
        "rmse_usd": 4362.51,
        "r2": 0.8964
    },
    "notes": "Metrics computed on held-out 20% test set before final refit on full data."
}
with open('../models/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved final_model.joblib and model_metadata.json")

Saved final_model.joblib and model_metadata.json
